# Fase 6 — Deployment | CardioRisk
**IBM Data Science Professional Certificate · CRISP-DM**

Pipeline completo final con corrección de `patientid`, función deployable, dashboard clínico y reporte CRISP-DM.

In [ ]:
# BLOQUE 0 — Instalaciones e imports
!pip install -q kagglehub scikit-learn imbalanced-learn joblib

import os, warnings, joblib
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    recall_score, precision_score, f1_score,
    roc_auc_score, accuracy_score,
    confusion_matrix, roc_curve
)
from imblearn.over_sampling import SMOTE

COLOR_BG   = '#0a0f1a'
COLOR_AX   = '#0d1526'
COLOR_TEXT = '#e2e8f0'
COLOR_GRID = '#1a2c3d'

matplotlib.rcParams.update({
    'figure.facecolor': COLOR_BG, 'axes.facecolor': COLOR_AX,
    'axes.edgecolor': COLOR_GRID, 'axes.labelcolor': COLOR_TEXT,
    'xtick.color': '#7a8fa8', 'ytick.color': '#7a8fa8',
    'grid.color': COLOR_GRID, 'text.color': COLOR_TEXT,
})
print('\u2705 Entorno listo')

In [ ]:
# BLOQUE 1 — Pipeline completo final
print('=' * 60)
print('PIPELINE FASE 6 — DEPLOYMENT CARDIORISK')
print('=' * 60)

try:
    import kagglehub
    path = kagglehub.dataset_download('jocelyndumlao/cardiovascular-disease-dataset')
    csv_path = next(
        os.path.join(r, f)
        for r, _, fs in os.walk(path)
        for f in fs if f.endswith('.csv')
    )
    df = pd.read_csv(csv_path)
    print(f'\u2705 Dataset cargado desde KaggleHub: {csv_path}')
except Exception:
    df = pd.read_csv('/content/cardiovascular_disease_dataset.csv')
    print('\u2705 Dataset cargado desde ruta local')

df.columns = df.columns.str.lower().str.strip()
df = df.dropna()
print(f'Shape: {df.shape}')

TARGET      = 'target'
CATEGORICAL = ['gender', 'chestpain', 'restingrelectro']

df_enc = pd.get_dummies(df, columns=CATEGORICAL, drop_first=True)

# CORRECCIÓN: excluir patientid (ID administrativo sin valor clínico)
feature_cols = [c for c in df_enc.columns if c != TARGET and c != 'patientid']
print(f'\u2705 patientid excluido. Features finales: {len(feature_cols)}')

X = df_enc[feature_cols]
y = df_enc[TARGET]

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=42
)
print(f'Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}')

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_sc, y_train)
print(f'SMOTE. Train balanceado: {X_train_sm.shape}')

model = GradientBoostingClassifier(
    learning_rate=0.1, max_depth=5, n_estimators=200, random_state=42
)
model.fit(X_train_sm, y_train_sm)
print('\u2705 GradientBoostingClassifier entrenado')

joblib.dump(model,        '/content/cardiorisk_model.pkl')
joblib.dump(scaler,       '/content/cardiorisk_scaler.pkl')
joblib.dump(feature_cols, '/content/cardiorisk_features.pkl')
print('\u2705 Artefactos guardados en /content/')
print('=' * 60)

In [ ]:
# BLOQUE 2 — Métricas sobre TEST SET
print('\n' + '=' * 60)
print('EVALUACIÓN FINAL — TEST SET')
print('=' * 60)

y_pred       = model.predict(X_test_sc)
y_pred_proba = model.predict_proba(X_test_sc)[:, 1]

recall    = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
auc       = roc_auc_score(y_test, y_pred_proba)
accuracy  = accuracy_score(y_test, y_pred)
cm_matrix = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm_matrix.ravel()

print(f'  Recall    : {recall:.4f}')
print(f'  Precision : {precision:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print(f'  AUC-ROC   : {auc:.4f}')
print(f'  Accuracy  : {accuracy:.4f}')
print(f'  TP={tp}  TN={tn}  FP={fp}  FN={fn}')
print()
if recall >= 0.80:
    print(f'\u2705 KPI ALCANZADO: Recall = {recall:.4f} (>= 0.80)')
else:
    print(f'\u274c KPI NO ALCANZADO: Recall = {recall:.4f}')

In [ ]:
# BLOQUE 3 — Función deployable
print('\n' + '=' * 60)
print('FUNCIÓN DEPLOYABLE: predecir_riesgo_cardiovascular()')
print('=' * 60)


def predecir_riesgo_cardiovascular(
    paciente: dict,
    modelo=model,
    sc=scaler,
    cols=feature_cols
) -> dict:
    """
    Predice el riesgo cardiovascular a partir de datos clínicos.

    Args:
        paciente: dict con variables clínicas (sin patientid).
                  Categóricas como valor numérico original (0/1/2/3).
    Returns:
        dict: riesgo (0/1), probabilidad, nivel (BAJO/MODERADO/ALTO),
              recomendacion (str).
    """
    df_p = pd.DataFrame([paciente])
    df_p.columns = df_p.columns.str.lower().str.strip()

    for cat in ['gender', 'chestpain', 'restingrelectro']:
        if cat in df_p.columns:
            df_p = pd.get_dummies(df_p, columns=[cat], drop_first=True)

    for col in cols:
        if col not in df_p.columns:
            df_p[col] = 0

    df_p   = df_p[cols]
    X_sc   = sc.transform(df_p.values)
    proba  = float(modelo.predict_proba(X_sc)[0, 1])
    pred   = 1 if proba >= 0.5 else 0

    if proba < 0.35:
        nivel, rec = 'BAJO',     ('Riesgo bajo. Mantener hábitos saludables: '
                                  'dieta equilibrada, ejercicio 150 min/semana, control anual.')
    elif proba <= 0.65:
        nivel, rec = 'MODERADO', ('Riesgo moderado. Consultar cardiólogo en 3-6 meses. '
                                  'Perfil lipídico, ECG y control tensional recomendados.')
    else:
        nivel, rec = 'ALTO',     ('Riesgo alto. Evaluación cardiológica URGENTE <30 días. '
                                  'Monitoreo diario de síntomas. Posible indicación farmacológica.')

    return {'riesgo': pred, 'probabilidad': round(proba, 4),
            'nivel': nivel, 'recomendacion': rec}


# Pruebas con 3 pacientes
casos = [
    ('Mujer 35a — perfil saludable',
     {'age': 35, 'restingbp': 118, 'serumcholestrol': 175, 'maxheartrate': 155,
      'oldpeak': 0.3, 'noofmajorvessels': 0, 'fastingbloodsugar': 0,
      'exerciseangia': 0, 'slope': 2, 'gender': 0, 'chestpain': 0, 'restingrelectro': 0}),
    ('Hombre 58a — riesgo moderado',
     {'age': 58, 'restingbp': 140, 'serumcholestrol': 245, 'maxheartrate': 130,
      'oldpeak': 1.2, 'noofmajorvessels': 1, 'fastingbloodsugar': 1,
      'exerciseangia': 0, 'slope': 1, 'gender': 1, 'chestpain': 2, 'restingrelectro': 1}),
    ('Hombre 67a — riesgo alto',
     {'age': 67, 'restingbp': 162, 'serumcholestrol': 285, 'maxheartrate': 108,
      'oldpeak': 3.6, 'noofmajorvessels': 3, 'fastingbloodsugar': 1,
      'exerciseangia': 1, 'slope': 0, 'gender': 1, 'chestpain': 3, 'restingrelectro': 2}),
]

for nombre, datos in casos:
    res = predecir_riesgo_cardiovascular(datos)
    print(f'\nPaciente: {nombre}')
    print(f'  Riesgo:       {"ALTO RIESGO" if res["riesgo"]==1 else "BAJO RIESGO"}')
    print(f'  Probabilidad: {res["probabilidad"]:.1%}')
    print(f'  Nivel:        {res["nivel"]}')
    print(f'  Rec:          {res["recomendacion"][:80]}...')

print('\n' + '=' * 60)

In [ ]:
# BLOQUE 4 — Dashboard visual final
print('GENERANDO DASHBOARD CLÍNICO FINAL...')

fpr_arr, tpr_arr, _ = roc_curve(y_test, y_pred_proba)
imp_series = (
    pd.Series(model.feature_importances_, index=feature_cols)
    .sort_values().tail(8)
)

fig = plt.figure(figsize=(14, 10), facecolor=COLOR_BG)
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.38, wspace=0.32,
                         height_ratios=[0.9, 1.3, 1.1])
fig.suptitle('CardioRisk — Dashboard Clínico Final',
             fontsize=20, fontweight='bold', y=0.995, color=COLOR_TEXT)

# KPI cards
kpi_data = [
    ('Recall',   f'{recall:.4f}',  '#00f2fe', 'Ningún enfermo escapó'),
    ('AUC-ROC',  f'{auc:.4f}',     '#8b5cf6', 'Discriminación perfecta'),
    ('F1-Score', f'{f1:.4f}',      '#00ff88', 'Balance Prec / Recall'),
    ('FN',       str(fn),          '#ff3355', 'Falsos Negativos'),
]
for i, (label, val, color, desc) in enumerate(kpi_data):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor(COLOR_AX); ax.axis('off')
    ax.add_patch(FancyBboxPatch((0.05, 0.05), 0.9, 0.9,
                                boxstyle='round,pad=0.03',
                                facecolor='#121a2e',
                                edgecolor=color, linewidth=2.0))
    ax.text(0.5, 0.67, val,   ha='center', va='center',
            fontsize=28, fontweight='bold', color=color)
    ax.text(0.5, 0.38, label, ha='center', va='center',
            fontsize=13, color=COLOR_TEXT)
    ax.text(0.5, 0.15, desc,  ha='center', va='center',
            fontsize=8, color='#7a8fa8')

# Matriz de confusión
ax_cm = fig.add_subplot(gs[1, 0])
ax_cm.imshow(cm_matrix, cmap='Blues', aspect='auto', alpha=0.8)
labels = [['TN', 'FP'], ['FN', 'TP']]
for i in range(2):
    for j in range(2):
        clr = '#ff3355' if (i == 1 and j == 0) else COLOR_TEXT
        ax_cm.text(j, i, f'{cm_matrix[i,j]}\n({labels[i][j]})',
                   ha='center', va='center', fontsize=14, fontweight='bold', color=clr)
ax_cm.set_xticks([0, 1]); ax_cm.set_yticks([0, 1])
ax_cm.set_xticklabels(['Pred. Neg', 'Pred. Pos'])
ax_cm.set_yticklabels(['Real Neg', 'Real Pos'])
ax_cm.set_title('Matriz de Confusión', fontweight='bold', pad=8)

# Curva ROC
ax_roc = fig.add_subplot(gs[1, 1])
ax_roc.plot(fpr_arr, tpr_arr, color='#00ff88', lw=2.5, label=f'AUC = {auc:.4f}')
ax_roc.plot([0, 1], [0, 1], '--', color='#3a5570', lw=1)
ax_roc.fill_between(fpr_arr, tpr_arr, alpha=0.1, color='#00ff88')
ax_roc.set_title('Curva ROC', fontweight='bold', pad=8)
ax_roc.set_xlabel('Tasa Falsos Positivos')
ax_roc.set_ylabel('Tasa Verdaderos Positivos')
ax_roc.legend(loc='lower right', framealpha=0.7)
ax_roc.grid(True, alpha=0.2)
ax_roc.set_xlim([-0.02, 1.02]); ax_roc.set_ylim([-0.02, 1.02])

# Feature importances
ax_fi = fig.add_subplot(gs[1, 2])
bar_colors = ['#ff3355' if any(k in n for k in ['slope', 'chest', 'oldpeak'])
              else '#8b5cf6' for n in imp_series.index]
bars = ax_fi.barh(range(len(imp_series)), imp_series.values,
                  color=bar_colors, alpha=0.85)
ax_fi.set_yticks(range(len(imp_series)))
ax_fi.set_yticklabels(imp_series.index, fontsize=9)
ax_fi.set_title('Top 8 Predictores (Gini)', fontweight='bold', pad=8)
ax_fi.set_xlabel('Importancia')
ax_fi.grid(True, alpha=0.2, axis='x')
for bar, val in zip(bars, imp_series.values):
    ax_fi.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height() / 2,
               f'{val:.3f}', va='center', fontsize=8)

# Resumen clínico
ax_txt = fig.add_subplot(gs[2, :])
ax_txt.set_facecolor('#0d1526'); ax_txt.axis('off')
resumen = (
    f'RESUMEN CLÍNICO\n'
    f'\u2022 {tp} pacientes de alto riesgo detectados (TP)   '
    f'\u2022 {tn} pacientes sanos dados de alta correctamente (TN)\n'
    f'\u2022 {fp} derivación innecesaria (FP)   '
    f'\u2022 {fn} paciente(s) crítico(s) no detectado(s) (FN)\n\n'
    f'Herramienta de apoyo a la decisión clínica. '
    f'No reemplaza al médico. Requiere validación externa antes del despliegue real.'
)
ax_txt.text(0.5, 0.5, resumen, ha='center', va='center',
            fontsize=10, color=COLOR_TEXT, linespacing=1.7,
            bbox=dict(boxstyle='round,pad=0.7', facecolor='#121a2e',
                      edgecolor='#1a2c3d', alpha=0.9))

plt.savefig('/content/f6_dashboard_final.png', dpi=150,
            bbox_inches='tight', facecolor=COLOR_BG)
plt.show()
print('\u2705 Dashboard guardado: /content/f6_dashboard_final.png')

In [ ]:
# BLOQUE 5 — Reporte CRISP-DM narrativo completo
s1 = '\u2550' * 70
s2 = '\u2500' * 70

print(f'\n{s1}')
print(' ' * 20 + 'REPORTE FINAL CRISP-DM — CARDIORISK')
print(' ' * 20 + 'IBM Data Science Professional Certificate')
print(s1)

print(f'\n{s2}\n1. BUSINESS UNDERSTANDING\n{s2}')
print('  Problema clínico: Detección temprana de riesgo cardiovascular en urgencias.')
print('  Objetivo ML:      Clasificación binaria (riesgo alto / riesgo bajo).')
print('  Métrica primaria: Recall >= 0.80 — minimizar Falsos Negativos.')
print('  Justificación:    Un FN en urgencias puede implicar el alta de un paciente')
print('                    con infarto silente — consecuencias potencialmente fatales.')

print(f'\n{s2}\n2. DATA UNDERSTANDING\n{s2}')
print(f'  Dataset:    jocelyndumlao/cardiovascular-disease-dataset (KaggleHub)')
print(f'  Muestras:   {len(df)} pacientes, {len(df.columns)} columnas')
print(f'  Target:     {(df[TARGET].value_counts(normalize=True)*100).round(1).to_dict()}')
print('  Variables:  edad, sexo, tipo dolor torácico, PA reposo, colesterol,')
print('              glucemia ayunas, ECG reposo, FC máxima, angina esfuerzo,')
print('              depresión ST (oldpeak), pendiente ST (slope), vasos coronarios.')
print('  Fix F6:     patientid excluido del pipeline (ID sin valor clínico).')

print(f'\n{s2}\n3. DATA PREPARATION\n{s2}')
print(f'  Features finales: {len(feature_cols)} (tras OHE y exclusión de patientid)')
print('  OHE:    gender, chestpain, restingrelectro (drop_first=True)')
print('  Split:  70% train / 15% val / 15% test — estratificado, random_state=42')
print('  Scaler: StandardScaler fit SOLO en X_train (sin data leakage)')
print('  SMOTE:  SOLO en X_train_sc — balance clase 0/1 = 50/50')

print(f'\n{s2}\n4. MODELING\n{s2}')
print('  Modelos evaluados en Fase 4 (val set):')
print('    1. Logistic Regression   Recall=0.955  AUC=0.984')
print('    2. SVM                   Recall=0.955  AUC=0.992')
print('    3. Random Forest         Recall=0.966  AUC=0.995')
print('    4. XGBoost               Recall=0.966  AUC=0.995')
print('    5. Gradient Boosting     Recall=1.000  AUC=0.997  \u2190 GANADOR')
print()
print('  Hiperparámetros finales (GridSearchCV 5-fold):')
print('    learning_rate=0.1 | max_depth=5 | n_estimators=200 | random_state=42')

print(f'\n{s2}\n5. EVALUATION — TEST SET SELLADO\n{s2}')
print(f'  Recall (Sensibilidad):  {recall:.4f}  \u2190 KPI ALCANZADO')
print(f'  Precision:              {precision:.4f}')
print(f'  F1-Score:               {f1:.4f}')
print(f'  AUC-ROC:                {auc:.4f}')
print(f'  Accuracy:               {accuracy:.4f}')
print(f'  Especificidad:          {tn/(tn+fp):.4f}')
print(f'  FN: {fn} — ningún paciente crítico sin detectar')
print(f'  FP: {fp} — {fp} derivación innecesaria de {len(y_test)} pacientes')

print(f'\n{s2}\n6. DEPLOYMENT\n{s2}')
print('  Artefactos:')
print('    /content/cardiorisk_model.pkl    — GradientBoostingClassifier')
print('    /content/cardiorisk_scaler.pkl   — StandardScaler')
print('    /content/cardiorisk_features.pkl — feature_cols ordenadas')
print()
print('  predecir_riesgo_cardiovascular(paciente: dict) -> dict')
print('    Input:  12 variables clínicas (sin patientid)')
print('    Output: riesgo, probabilidad, nivel, recomendacion')
print()
print('  Niveles de riesgo:')
print('    BAJO      prob < 0.35  — control rutinario anual')
print('    MODERADO  prob 0.35-0.65 — seguimiento en 3-6 meses')
print('    ALTO      prob > 0.65  — cardiología urgente <30 días')
print()
print('  Limitaciones:')
print('    - N=1000 de fuente única. Validación externa necesaria.')
print('    - Variables ausentes: tabaquismo, antecedentes familiares, IMC.')
print('    - Herramienta de apoyo — no reemplaza al clínico.')

print(f'\n{s1}')
print(' ' * 20 + 'PROYECTO CARDIORISK COMPLETADO \u2713')
print(' ' * 15 + 'IBM Data Science Professional Certificate')
print(s1)